In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw8.ipynb")

# CPSC 330 - Applied Machine Learning

## Homework 8: Introduction to Computer vision and Time Series

**Due date: See [deliverable due dates](https://ubc-cs.github.io/cpsc330-2025W2/#deliverable-due-dates-tentative)**.

## Imports

In [ ]:
from hashlib import sha1

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## Instructions
rubric={points}

You will earn points for following these instructions and successfully submitting your work on Gradescope.  

### Group wotk instructions

**You may work with a partner on this homework and submit your assignment as a group.** Below are some instructions on working as a group.  
- The maximum group size is 2.
  
- Use group work as an opportunity to collaborate and learn new things from each other. 
- Be respectful to each other and make sure you understand all the concepts in the assignment well. 
- It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. 
- You can find the instructions on how to do group submission on Gradescope [here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members).
- If you would like to use late tokens for the homework, all group members must have the necessary late tokens available. Please note that the late tokens will be counted for all members of the group.   


### General submission instructions

- Please **read carefully
[Use of Generative AI policy](https://ubc-cs.github.io/cpsc330-2025W2/syllabus#use-of-generative-ai-in-the-course)** before starting the homework assignment. 
- **Run all cells before submitting:** Go to `Kernel -> Restart Kernel and Clear All Outputs`, then select `Run -> Run All Cells`. This ensures your notebook runs cleanly from start to finish without errors.
  
- **Submit your files on Gradescope.**  
   - Upload only your `.ipynb` file **with outputs displayed** and any required output files.
     
   - Do **not** submit other files from your repository.  
   - If you need help, see the [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/).  
- **Check that outputs render properly.**  
   - Make sure all plots and outputs appear in your submission.
     
   - If your `.ipynb` file is too large and doesn't render on Gradescope, also upload a PDF or HTML version so the TAs can view your work.  
- **Keep execution order clean.**  
   - Execution numbers must start at "1" and increase in order.
     
   - Notebooks without visible outputs may not be graded.  
   - Out-of-order or missing execution numbers may result in mark deductions.  
- **Follow course submission guidelines:** Review the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions) for detailed guidance on completing and submitting assignments. 
   
</div>

_Points:_ 2

<!-- END QUESTION -->

<br><br>

## Exercise 1: time series prediction

In this exercise we'll be looking at a [dataset of avocado prices](https://www.kaggle.com/neuromusic/avocado-prices). You should start by downloading the dataset and storing it under the `data` folder. We will be forcasting average avocado price for the next week. 

In [ ]:
df = pd.read_csv("data/avocado.csv", parse_dates=["Date"], index_col=0)
df.head()

In [ ]:
df.shape

In [ ]:
df["Date"].min()

In [ ]:
df["Date"].max()

It looks like the data ranges from the start of 2015 to March 2018 (~2 years ago), for a total of 3.25 years or so. Let's split the data so that we have a 6 months of test data.

In [ ]:
split_date = '20170925'
df_train = df[df["Date"] <= split_date]
df_test  = df[df["Date"] >  split_date]

In [ ]:
assert len(df_train) + len(df_test) == len(df)

<br><br>

<!-- BEGIN QUESTION -->

### 1.1 How many time series? 
rubric={points:4}

In the [Rain in Australia](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package) dataset from lecture demo, we had different measurements for each Location. 

We want you to consider this for the avocado prices dataset. For which categorical feature(s), if any, do we have separate measurements? Justify your answer by referencing the dataset.

<div class="alert alert-warning">

Solution_1.1
    
</div>

_Points:_ 4

_Type your answer here, replacing this text._

In [ ]:
df['Date'].value_counts().gt(1).any()

In [ ]:
df.duplicated(['Date', 'region', 'type']).any()

I first checked if there were any duplicate dates, as this would indicate if separate measurements are needed. As there are, I used a process of elimination to find the minimal combination needed for unique rows. If we combine 'Date', 'region', and 'type' together, we are able to separate measurements, given that we can get one row per date or get one row if we change just one of them. This can be seen in the dataset, as we find that there is more than one row with the same date, which means we have to combine other features to make it unique. When checking for duplicates with the relationship of 'Date', 'region', and 'type', we find there are no duplicates. 

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.2 Equally spaced measurements? 
rubric={points:4}

In the Rain in Australia dataset, the measurements were generally equally spaced but with some exceptions. How about with this dataset? Justify your answer by referencing the dataset.

<div class="alert alert-warning">

Solution_1.2
    
</div>

_Points:_ 4

Note that the following method was adapted from tutorial 8.

In [ ]:
def time_spacing_distribution(df, region, avo_type):
    df['Date'] = pd.to_datetime(df['Date'])
    region_type_data = df[(df['region'] == region) & (df['type'] == avo_type)]
    
    if region_type_data.empty:
        print(f"No data available for region and type: {region, avo_type}")
        return
    
    time_diffs = region_type_data['Date'].sort_values().diff().dropna()
    value_counts = time_diffs.value_counts().sort_index()
    print(f"Time spacing counts for {region} + {avo_type}: {value_counts}\n")

In [ ]:
types = df['type'].unique()
regions = df['region'].unique()

for t in types:
    for r in regions:
        time_spacing_distribution(df, r, t)

This dataset is nearly equally spaced but we have one exception with WestTexNewMexico + organic, where the count nearly matches all others except that there are three missing counts for 7 days, one count of 14 days and one count of 21 days. This can be seen in the above count. As there is a difference in the time spacing counts, this indicates the measurements are not perfectly equally spaced.

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Interpreting regions 
rubric={points:4}

In the Rain in Australia dataset, each location was a different place in Australia. For this dataset, look at the names of the regions. Do you think the regions are also all distinct, or are there overlapping regions? Justify your answer by referencing the data.

<div class="alert alert-warning">

Solution_1.3
    
</div>

_Points:_ 4

_Type your answer here, replacing this text._

In [ ]:
regions

There is definite overlapping regions as there is "TotalUS" and multiple states found in the region. Additionally, there are cities included (ie. SanFrancisco) and states (ie. California), indicating overalapping regions.

<!-- END QUESTION -->

<br><br>

We will use the entire dataset despite any location-based weirdness uncovered in the previous part.

We will be trying to forecast the avocado price. The function below is adapted from the lecture.

In [ ]:
import pandas as pd


def create_lag_feature(
    df: pd.DataFrame,
    orig_feature: str,
    lag: int,
    groupby: list[str],
    new_feature_name: str | None = None,
    clip: bool = False,
) -> pd.DataFrame:
    """
    Create a lagged (or ahead) version of a feature, optionally per group.

    Assumes df is already sorted by time within each group and has unique indices.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset.
    orig_feature : str
        Name of the column to lag.
    lag : int
        The lag:
          - negative ? values from the past (t-1, t-2, ...)
          - positive ? values from the future (t+1, t+2, ...)
    groupby : list of str
        Column(s) to group by if df contains multiple time series.
    new_feature_name : str, optional
        Name of the new column. If None, a name is generated automatically.
    clip : bool, default False
        If True, drop rows where the new feature is NaN.

    Returns
    -------
    pd.DataFrame
        A new dataframe with the additional column added.
    """
    if lag == 0:
        raise ValueError("lag cannot be 0 (no shift). Use the original feature instead.")

    # Default name if not provided
    if new_feature_name is None:
        if lag < 0:
            new_feature_name = f"{orig_feature}_lag{abs(lag)}"
        else:
            new_feature_name = f"{orig_feature}_ahead{lag}"

    df = df.copy()

    # Map your convention (negative=past, positive=future) to pandas shift
    # pandas: shift(+k) ? past, shift(-k) ? future
    periods = abs(lag) if lag < 0 else -lag

    df[new_feature_name] = (
        df.groupby(groupby, sort=False)[orig_feature]
          .shift(periods)
    )

    if clip:
        df = df.dropna(subset=[new_feature_name])

    return df


We first sort our dataframe properly:

In [ ]:
df_sort = df.sort_values(by=["region", "type", "Date"]).reset_index(drop=True)
df_sort

We then call `create_lag_feature`. This creates a new column in the dataset `AveragePriceNextWeek`, which is the following week's `AveragePrice`. We have set `clip=True` which means it will remove rows where the target would be missing.

In [ ]:
df_hastarget = create_lag_feature(df_sort, "AveragePrice", +1, ["region", "type"], "AveragePriceNextWeek", clip=True)
df_hastarget

Our goal is to predict `AveragePriceNextWeek`. 

Let's split the data:

In [ ]:
df_train = df_hastarget[df_hastarget["Date"] <= split_date]
df_test  = df_hastarget[df_hastarget["Date"] >  split_date]

<br><br>

<!-- BEGIN QUESTION -->

### 1.4 `AveragePrice` baseline 
rubric={points}

Soon we will want to build some models to forecast the average avocado price a week in advance. Before we start with any ML though, let's try a baseline. Previously we used `DummyClassifier` or `DummyRegressor` as a baseline. This time, we'll do something else as a baseline: we'll assume the price stays the same from this week to next week. So, we'll set our prediction of "AveragePriceNextWeek" exactly equal to "AveragePrice", assuming no change. That is kind of like saying, "If it's raining today then I'm guessing it will be raining tomorrow". This simplistic approach will not get a great score but it's a good starting point for reference. If our model does worse that this, it must not be very good. 

Using this baseline approach, what $R^2$ do you get on the train and test data?

<div class="alert alert-warning">

Solution_1.4
    
</div>

_Points:_ 4

_Type your answer here, replacing this text._

In [ ]:
train_r2 = ...

In [ ]:
test_r2 = ...

In [ ]:
df = df.sort_values(['region', 'type', 'Date'])

df['AveragePriceNextWeek'] = (
    df.groupby(['region', 'type'])['AveragePrice']
      .shift(-1)
)

df = df.dropna(subset=['AveragePriceNextWeek'])

In [ ]:
train_r2 = r2_score(df_train['AveragePriceNextWeek'], df_train['AveragePrice'])
test_r2 = r2_score(df_test['AveragePriceNextWeek'], df_test['AveragePrice'])

In [ ]:
train_r2

In [ ]:
test_r2

In [ ]:
assert not train_r2 is None, "Are you using the correct variable name?"
assert not test_r2 is None, "Are you using the correct variable name?"
assert sha1(str(round(train_r2, 3)).encode('utf8')).hexdigest() == 'b1136fe2a8918904393ab6f40bfb3f38eac5fc39', "Your training score is not correct. Are you using the right features?"
assert sha1(str(round(test_r2, 3)).encode('utf8')).hexdigest() == 'cc24d9a9b567b491a56b42f7adc582f2eefa5907', "Your test score is not correct. Are you using the right features?"

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.5 Forecasting average avocado price
rubric={points:10}

Now that the baseline is done, let's build some models to forecast the average avocado price a week later. Experiment with a few approachs for encoding the date. Justify the decisions you make. Which approach worked best? Report your test score and briefly discuss your results.

Benchmark: you should be able to achieve $R^2$ of at least 0.79 on the test set. I got to 0.80, but not beyond that. Let me know if you do better!

Note: because we only have 2 splits here, we need to be a bit wary of overfitting on the test set. Try not to test on it a ridiculous number of times. If you are interested in some proper ways of dealing with this, see for example sklearn's [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html), which is like cross-validation for time series data.

<div class="alert alert-warning">

Solution_1.5
    
</div>

_Points:_ 10

_Type your answer here, replacing this text._

Note that the following pre-processing method was adapted from tutorial 8.

In [ ]:
df_train['month'] = df_train['Date'].dt.month
df_train['week'] = df_train['Date'].dt.isocalendar().week

df_test['month'] = df_test['Date'].dt.month
df_test['week'] = df_test['Date'].dt.isocalendar().week

In [ ]:
numeric_features = [
    "AveragePrice",
    "Total Volume",
    "4046",
    "4225",
    "4770",
    "Total Bags",
    "Small Bags",
    "Large Bags",
    "XLarge Bags",
    "year",
    "month",
    "week"
]
categorical_features = [
    "type",
    "region"
]
drop_features = []
target = ["AveragePriceNextWeek"]

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
def preprocess_features(
    train_df,
    test_df,
    numeric_features,
    categorical_features,
    drop_features,
    target
):

    all_features = set(numeric_features + categorical_features + drop_features + target)
    if set(train_df.columns) != all_features:
        print("Missing columns", set(train_df.columns) - all_features)
        print("Extra columns", all_features - set(train_df.columns))
        raise Exception("Columns do not match")

    numeric_transformer = make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler()
    )
    categorical_transformer = make_pipeline(
        SimpleImputer(strategy="constant", fill_value="missing"),
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
    )

    preprocessor = make_column_transformer(
        (numeric_transformer, numeric_features),
        (categorical_transformer, categorical_features),
        ("drop", drop_features),
    )
    preprocessor.fit(train_df)
    ohe_feature_names = (
        preprocessor.named_transformers_["pipeline-2"]
        .named_steps["onehotencoder"]
        .get_feature_names_out(categorical_features)
        .tolist()
    )
    new_columns = numeric_features + ohe_feature_names

    X_train_enc = pd.DataFrame(
        preprocessor.transform(train_df), index=train_df.index, columns=new_columns
    )
    X_test_enc = pd.DataFrame(
        preprocessor.transform(test_df), index=test_df.index, columns=new_columns
    )

    y_train = train_df["AveragePriceNextWeek"]
    y_test = test_df["AveragePriceNextWeek"]

    return X_train_enc, y_train, X_test_enc, y_test, preprocessor

In [ ]:
df_train = df_train.drop(columns=["Date"])
df_test = df_test.drop(columns=["Date"])

X_train_enc, y_train, X_test_enc, y_test, preprocessor = preprocess_features(
    df_train,
    df_test,
    numeric_features,
    categorical_features,
    drop_features, target
)

In [ ]:
X_train_enc.head()

Model 1: Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

lr = LinearRegression()
lr.fit(X_train_enc, y_train)

y_pred_lr = lr.predict(X_test_enc)
r2_lr = r2_score(y_test, y_pred_lr)

Model 2: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_enc, y_train)

y_pred_rf = rf.predict(X_test_enc)
r2_rf = r2_score(y_test, y_pred_rf)

Model 3: Gradient Boosting Regressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gbr.fit(X_train_enc, y_train)

y_pred_gbr = gbr.predict(X_test_enc)
r2_gbr = r2_score(y_test, y_pred_gbr)

In [ ]:
print("Linear Regression R^2:", r2_lr)
print("Random Forest R^2:", r2_rf)
print("Gradient Boosting R^2:", r2_gbr)

To approach the Date encoding, I used the month and week, as I believe this captures a more accurate version of avocado pricing. As avocados are grown best at specific times of the year, I believe the seasonal date will impact the prices, ie. a price increase during the off-season. Therefore, I thought this way of encoding the date seasonally would make the most sense.

I first preprocessed the data, as we have missing data, categorical features and numeric features. I wanted to standardize the values as well, mitigating any extremes in the data.

I trained several models, including linear regression, random forest, and gradient boosting. Each model gave a fairly strong performance, with 0.776, 0.784 and 0.794, respectively. However, testing on gradient boosting was the strongest, with an R^2 score of 0.794. The difference in score makes sense as linear regression tends not to be able to capture nonlinear patterns (ie. Date). Random forest improved performance by modelling interactions between features, but gradient boosting was the best since it picked up the patterns more accurately, while fixing its own mistakes.

It is possible that there may be slight overfitting as the test set was run multiple times; however, because only three models ran the test set, I do not believe a ridiculous number of runs were used to cause an overfitting issue.

Ultimately, gradient boosting provided the best model, as we did not overfit the test data, and it was able to work with complex data, achieving the highest R^2 score out of the three models.

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 2: Short answer questions

<!-- BEGIN QUESTION -->

### 2.1 Time series

rubric={points:6}

The following questions pertain to Lecture 20 on time series data:

1. Sometimes a time series has missing time points or, worse, time points that are unequally spaced in general. Give an example of a real world situation where the time series data would have unequally spaced time points.
2. In class we discussed two approaches to using temporal information: encoding the date as one or more features, and creating lagged versions of features. Which of these (one/other/both/neither) two approaches would struggle with unequally spaced time points? Briefly justify your answer.
3. When studying time series modeling, we explored several ways to encode date information as a feature for the citibike dataset. When we used time of day as a numeric feature, the Ridge model was not able to capture the periodic pattern. Why? How did we tackle this problem? Briefly explain.

<div class="alert alert-warning">

Solution_2.1
    
</div>

_Points:_ 6

1. A real-world example would be in stock trading. As anyone can buy or sell stocks at any time of the day, consistent transactions do not always occur at regular intervals, and would cause unequally spaced time points.
   
2. I believe lagged versions of features would have trouble with unequally spaced time points. This is because a lag/shift assumes a consistent time interval between observations. This means if yesterday?s data is missing, the lag period dependent on yesterday's data could reference the wrong date, introduce bias, and/or misinterpret time-interval dependencies. Encoding the date would be fine, as it is not dependent on equal intervals.
   
3. Ridge was unable to capture the periodic pattern as it is a linear model and assumes a linear relationship between dependent and independent variables. If time-of-day is a numeric feature, it cannot capture a cyclic pattern for periodic data, like how hour 0 = hour 24. We tackled this problem by attempting to encode it as a categorical feature and adding interaction features. We did the former with one-hot encoding by converting the hour of day and day of week into separate binary columns, allowing the model to capture the patterns better. Additionally, we made combinations with interaction features, allowing the model to learn patterns that depend on both day and hour. These strategies allowed the model to capture the complex patterns better, increasing the overall R^2 score.

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.2 Computer vision 
rubric={points:6}

The following questions pertain to the lecture on multiclass classification and introduction to computer vision. 

1. How many parameters (coefficients and intercepts) will `sklearn`?s `LogisticRegression()` model learn for a four-class classification problem, assuming that you have 10 features? Briefly explain your answer.
2. In Lecture 19, we briefly discussed how neural networks are sort of like `sklearn`'s pipelines, in the sense that they involve multiple sequential transformations of the data, finally resulting in the prediction. Why was this property useful when it came to transfer learning?
3. Imagine that you have a small dataset with ~1000 images containing pictures and names of 50 different Computer Science faculty members from UBC. Your goal is to develop a reasonably accurate multi-class classification model for this task. Describe which model/technique you would use and briefly justify your choice in one to three sentences.

<div class="alert alert-warning">

Solution_2.2
    
</div>

_Points:_ 6

1. For 4 classes and 10 features, each class gets its own set of coefficients plus an intercept. This means there will be 40 coefficients (10 features * 4 classes) and 4 intercepts (1 per class), totalling 44 parameters for LogisticRegression() model to learn.

2. Neural networks inehrently have a hierarchal or layered-style feature for data. This property is useful for transfer learning because general features can be captured and reused, especially the early layers. Using a pre-trained network, you only need to retrain the final layers for each unique task. Overall, it saves computation and improves accuracy on small datasets.

3. I would use transfer learning with a neural network. This is because the dataset is small (~20 photos per faculty member), so overfitting is a big issue. Using a pre-trained model leverages learned general features, and you only need to finetune the last few layers to classify the 50 faculty members.

<!-- END QUESTION -->

<br><br>

Before submitting your assignment, please make sure you have followed all the instructions in the Submission Instructions section at the top. 

Here is a quick checklist before submitting: 

- [ ] Restart kernel, clear outputs, and run all cells from top to bottom.  
- [ ] `.ipynb` file runs without errors and contains all outputs.  
- [ ] Only `.ipynb` and required output files are uploaded (no extra files).  
- [ ] Execution numbers start at **1** and are in order.  
- [ ] If `.ipynb` is too large and doesn't render on Gradescope, also upload a PDF/HTML version.  
- [ ] Reviewed the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions).  

![](img/eva-well-done.png)